# CAPTCHA Patch Selection Demo
This notebook demonstrates the principle of image-based CAPTCHAs. 
1. It loads a full-size image.
2. It extracts random patches and presents them in a grid.
3. You (the user) select patches based on a prompt (e.g., "Select all traffic signs").
4. Your selections are accumulated and visualized as a heatmap overlay on the original image, revealing the "segmented" areas of interest over time.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import ipywidgets as widgets
from IPython.display import display
import io
import random

# random.seed(42)

## Load Image & Initialize State
We load the target image and initialize the heatmap accumulator.

In [2]:
# Configuration
IMAGE_PATH = 'verkehr.jpg' 
PATCH_SIZE = (300, 300)     
GRID_DIMS = (3, 3)      

# Load Image
full_image = Image.open(IMAGE_PATH).convert('RGB')
full_image = full_image.resize((1200, 800))
full_image_np = np.array(full_image)
print(f"Loaded image: {IMAGE_PATH} with shape {full_image_np.shape}")

# Initialize Heatmap (same dimensions as image, single channel)
heatmap = np.zeros(full_image_np.shape[:2], dtype=np.float32)
selection_count = 0

Loaded image: verkehr.jpg with shape (800, 1200, 3)


## Helper Functions
Functions to extract patches, convert images for widgets, and generate the heatmap overlay.

In [3]:
def get_random_patch(image_np, size=(100, 100)):
    """Extracts a random patch from the image."""
    h, w, _ = image_np.shape
    ph, pw = size
    # Ensure patch fits in image
    if h < ph or w < pw:
        return np.zeros((ph, pw, 3), dtype=np.uint8), 0, 0
        
    x = random.randint(0, w - pw)
    y = random.randint(0, h - ph)
    patch = image_np[y:y+ph, x:x+pw]
    return patch, x, y

def np_to_b64(img_np):
    """Converts a numpy image to bytes for display in ipywidgets."""
    img = Image.fromarray(img_np)
    buf = io.BytesIO()
    img.save(buf, format='PNG')
    return buf.getvalue()

def overlay_heatmap(img_np, heatmap):
    """Overlays the accumulated heatmap on the original image."""
    # Normalize heatmap to 0-1 range for colormap
    max_val = np.max(heatmap)
    if max_val > 0:
        norm_heatmap = heatmap / max_val
    else:
        norm_heatmap = heatmap
    
    # Apply colormap
    heatmap_colored = plt.cm.jet(norm_heatmap)[:, :, :3]
    heatmap_colored = (heatmap_colored * 255).astype(np.uint8)
    
    # Blend images
    alpha = 0.4 
    overlay = (img_np * (1 - alpha) + heatmap_colored * alpha).astype(np.uint8)
    mask = norm_heatmap == 0
    overlay[mask] = img_np[mask]
    
    return overlay

## Interactive Interface
Run the cell below to start the interactive session. 
1. **Select** the patches that match the criteria.
2. Click **Proceed / Next Grid** to submit your selection and get new patches.
3. Observe the **Accumulated Heatmap** updating below.

In [4]:
# UI Elements
grid_container = widgets.VBox()
img_heatmap = widgets.Image()
lbl_heatmap_title = widgets.HTML()
btn_proceed = widgets.Button(
    description="Proceed / Next Grid",
    button_style='success', # 'success', 'info', 'warning', 'danger' or ''
    layout=widgets.Layout(width='200px', margin='20px 0px 0px 0px')
)
lbl_info = widgets.HTML(value="<h3>Prompt: Select all patches containing <b>Traffic Lights</b></h3>")

# State variables for the current grid
current_patches_coords = []
img_widgets = []
checkboxes = []

def generate_grid():
    """Generates a new grid of random patches and updates the UI."""
    global current_patches_coords, img_widgets, checkboxes
    current_patches_coords = []
    
    n_patches = GRID_DIMS[0] * GRID_DIMS[1]
    
    # Initialize widgets if not already done
    if not img_widgets:
        for _ in range(n_patches):
            img_widget = widgets.Image(format='png', width=PATCH_SIZE[1], height=PATCH_SIZE[0])
            chk_widget = widgets.Checkbox(value=False, description='Select', indent=False)
            img_widgets.append(img_widget)
            checkboxes.append(chk_widget)
    
    for i in range(n_patches):
        patch, x, y = get_random_patch(full_image_np, PATCH_SIZE)
        current_patches_coords.append((x, y))
        
        img_bytes = np_to_b64(patch)
        img_widgets[i].value = img_bytes
        checkboxes[i].value = False
    
    # Arrange in a grid layout (VBox of HBoxes)
    rows = []
    for i in range(GRID_DIMS[0]):
        row_items = []
        for j in range(GRID_DIMS[1]):
            idx = i * GRID_DIMS[1] + j
            item = widgets.VBox([img_widgets[idx], checkboxes[idx]], layout=widgets.Layout(align_items='center', margin='5px'))
            row_items.append(item)
        rows.append(widgets.HBox(row_items))
    
    grid_container.children = rows

def on_proceed_clicked(b):
    """Callback for the Proceed button."""
    global heatmap, selection_count
    
    # 1. Update heatmap based on current selections
    selections_made = False
    for i, chk in enumerate(checkboxes):
        if chk.value:
            x, y = current_patches_coords[i]
            ph, pw = PATCH_SIZE
            # Increment the heatmap region corresponding to the selected patch
            heatmap[y:y+ph, x:x+pw] += 1.0
            selections_made = True
    
    if selections_made:
        selection_count += 1
    
    # 2. Refresh Grid with new patches
    generate_grid()
    
    # 3. Update Heatmap Display
    lbl_heatmap_title.value = f"<h3>Accumulated Heatmap (Selections: {selection_count})</h3>"
    overlayed = overlay_heatmap(full_image_np, heatmap)
    img = Image.fromarray(overlayed)
    buf = io.BytesIO()
    img.save(buf, format='PNG')
    img_heatmap.value = buf.getvalue()

btn_proceed.on_click(on_proceed_clicked)

# Initial Display
generate_grid()
lbl_heatmap_title.value = f"<h3>Accumulated Heatmap (Selections: {selection_count})</h3>"

# Layout the main application
app_layout = widgets.VBox([
    lbl_info, 
    grid_container, 
    btn_proceed, 
    # widgets.HTML("<hr><h3>Accumulated Result:</h3>"),
    lbl_heatmap_title,
    img_heatmap
])

display(app_layout)